## Index metadonnees — recherche hybride (visuel + texte)

Constat : la recherche vectorielle ne compare jusqu'ici que des **images**
(SigLIP) — une requete comme "editions imprimees a Lyon" ou "gravures de Solis"
ne peut pas bien fonctionner, puisque SigLIP encode ce qu'une image montre, pas
des faits bibliographiques.

Teste et rejete : reutiliser SigLIP pour comparer du texte a du texte (metadonnees).
Resultat sur 4 requetes de validation : "Cadmus combat un serpent" faisait remonter
en premier un document *Cain et Abel* au lieu du document *Cadmus et le dragon* —
SigLIP n'a jamais ete entraine pour la similarite texte-texte, seulement image-texte.

Solution retenue : un modele d'embedding texte dedie, **`intfloat/multilingual-e5-small`**
(entraine pour la recherche par similarite texte-texte), pour indexer une description
textuelle des metadonnees de chaque illustration. Meme test de validation : les 4
requetes trouvent le bon document en premier.

In [1]:
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer

RACINE = Path("../../").resolve()
DOSSIER_VECTOR_DB = RACINE / "data" / "vector_bases"

modele_e5 = SentenceTransformer("intfloat/multilingual-e5-small")
print("Modele e5 charge")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modele e5 charge


### 1. Construction du texte metadonnees par illustration

Un gabarit different par source : pour Ovide, on assemble titre/ville/graveur/
technique/theme ; pour les Bibles (qui n'ont que le theme comme metadonnee),
`THEME_MOTS_CLES_BIBLE` enrichit chaque theme avec des mots-cles narratifs
(noms propres, objets), pour eviter qu'un intitule trop court ne se confonde
avec un theme voisin (voir Limites connues dans `docs/mini_rag_iconographique.md`).

In [2]:
# Les themes bibliques sont des labels courts ("Cain et Abel", "Tentation"...)
# sans autre metadonnee associee (pas de titre/ville/graveur par illustration
# pour les Bibles) : le texte envoye a e5 etait donc juste "Illustration
# biblique, theme X", trop pauvre pour discriminer des recits proches dans
# l'espace d'embedding (ex. "Adam et Arbre" se rapprochait davantage de
# "Cain et Abel" que de "Tentation", faute de mots-cles narratifs communs).
# Ce champ descriptif ajoute les noms propres et objets caracteristiques de
# chaque scene, pour que la similarite texte-texte porte sur le contenu
# iconographique reel plutot que sur le seul intitule du theme.
THEME_MOTS_CLES_BIBLE = {
    "Babel": "tour de Babel, construction, confusion des langues, ouvriers",
    "Cain et Abel": "Cain, Abel, meurtre, sacrifice, freres, offrande",
    "Creation EVE": "creation d'Eve, cote d'Adam, Dieu createur, jardin d'Eden",
    "Creation du MONDE": "creation du monde, Dieu createur, ciel, terre, separation des elements",
    "Creation HOMME": "creation d'Adam, Dieu createur, souffle de vie, glaise",
    "Creation HORS ENTRAINEMENT": "scene de la creation, Genese",
    "Deluge": "deluge, arche de Noe, inondation, animaux, pluie",
    "Deluge AVANT": "avant le deluge, annonce du deluge, Noe construit l'arche",
    "Deluge APRES": "apres le deluge, arc-en-ciel, alliance, Noe sort de l'arche",
    "Chasses du paradis": "expulsion du paradis, Adam et Eve chasses du jardin d'Eden, ange a l'epee",
    "Sacrifice ABRAHAM": "sacrifice d'Abraham, Isaac, ange, belier, autel",
    "Tentation": "tentation, Adam, Eve, arbre, serpent, pomme, fruit defendu, jardin d'Eden",
}


def construire_texte_metadonnees(row):
    morceaux = []
    if row["source"] == "bible":
        if pd.notna(row.get("theme")):
            theme = row["theme"]
            morceaux.append(f"Illustration biblique, theme {theme}")
            mots_cles = THEME_MOTS_CLES_BIBLE.get(theme)
            if mots_cles:
                morceaux.append(mots_cles)
        else:
            morceaux.append("Illustration biblique")
    else:
        if pd.notna(row.get("theme_precis")):
            morceaux.append(str(row["theme_precis"]))
        elif pd.notna(row.get("theme")):
            morceaux.append(str(row["theme"]))
        if pd.notna(row.get("titre")):
            morceaux.append(f"edition : {row['titre']}")
        if pd.notna(row.get("ville")):
            morceaux.append(f"ville : {row['ville']}")
        if pd.notna(row.get("annee")):
            morceaux.append(f"annee : {row['annee']}")
        if pd.notna(row.get("graveur")):
            morceaux.append(f"graveur : {row['graveur']}")
        if pd.notna(row.get("technique")):
            morceaux.append(f"technique : {row['technique']}")
        if pd.notna(row.get("type_iconographique")):
            morceaux.append(f"type iconographique : {row['type_iconographique']}")
        if pd.notna(row.get("description_planche")):
            morceaux.append(str(row["description_planche"])[:500])
    return ". ".join(morceaux) if morceaux else "illustration sans metadonnees connues"

### 2. Calcul des embeddings et sauvegarde (bibles + ovide)

In [3]:
for nom_fichier, source in [("bibles_siglip.pkl", "bible"), ("ovide_corpus_complet_siglip.pkl", "ovide")]:
    chemin = DOSSIER_VECTOR_DB / nom_fichier
    base = pd.read_pickle(chemin)
    base["source"] = source

    textes = base.apply(construire_texte_metadonnees, axis=1)
    print(f"{nom_fichier} — exemple de texte : {textes.iloc[0][:150]}")

    embeddings = modele_e5.encode(
        [f"passage: {t}" for t in textes],
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=64,
    )
    base["embedding_metadonnees"] = list(embeddings)
    base = base.drop(columns=["source"])
    base.to_pickle(chemin)
    print(f"{nom_fichier} : {len(base)} embeddings metadonnees sauvegardes\n")

bibles_siglip.pkl — exemple de texte : Illustration biblique, theme Babel. tour de Babel, construction, confusion des langues, ouvriers


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

bibles_siglip.pkl : 372 embeddings metadonnees sauvegardes



ovide_corpus_complet_siglip.pkl — exemple de texte : creation_monde. edition : Trois Premiers livres de la Métamorphose d'Ovide. ville : Lyon. annee : 1556. graveur : Eskrich, Pierre. technique : bois. t


Batches:   0%|          | 0/35 [00:00<?, ?it/s]

ovide_corpus_complet_siglip.pkl : 2191 embeddings metadonnees sauvegardes



### 3. Verification — recherche hybride sur quelques requetes reelles

Reimplementation volontairement simplifiee (visuel + metadonnees, sans le
filtre de correspondance exacte) pour valider l'apport de l'index metadonnees
isolement. La version complete utilisee par l'appli vit dans `rag_utils.py`
(`rechercher_hybride()`).

In [4]:
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, AutoProcessor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_bibles = pd.read_pickle(DOSSIER_VECTOR_DB / "bibles_siglip.pkl")
base_bibles["source"] = "bible"
base_ovide = pd.read_pickle(DOSSIER_VECTOR_DB / "ovide_corpus_complet_siglip.pkl")
base_ovide["source"] = "ovide"

COLS_COMMUNES = ["chemin", "source", "theme", "embedding", "embedding_metadonnees", "url_page", "url_image"]
COLS_OVIDE_EXTRA = ["titre", "ville", "annee", "graveur", "technique",
                     "type_iconographique", "famille_iconographique",
                     "theme_precis", "description_planche"]
index = pd.concat([
    base_bibles.reindex(columns=COLS_COMMUNES + COLS_OVIDE_EXTRA),
    base_ovide.reindex(columns=COLS_COMMUNES + COLS_OVIDE_EXTRA),
], ignore_index=True)

X_index = np.array(index["embedding"].tolist())
X_index_meta = np.array(index["embedding_metadonnees"].tolist())
print(f"Index : {len(index)} illustrations, X_index {X_index.shape}, X_index_meta {X_index_meta.shape}")

processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
model_siglip = AutoModel.from_pretrained("google/siglip-base-patch16-224").to(DEVICE).eval()


def embed_texte_siglip(requete):
    inputs = processor(text=[requete], padding="max_length", return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        vec = model_siglip.get_text_features(**inputs).pooler_output
    return vec.cpu().numpy()[0]


def rechercher_hybride(requete, k=5, k_rrf=60):
    v_visuel = embed_texte_siglip(requete)
    v_meta = modele_e5.encode([f"query: {requete}"], normalize_embeddings=True)[0]

    sims_visu = cosine_similarity(v_visuel.reshape(1, -1), X_index)[0]
    sims_meta = cosine_similarity(v_meta.reshape(1, -1), X_index_meta)[0]

    rangs_visu = (-sims_visu).argsort().argsort() + 1
    rangs_meta = (-sims_meta).argsort().argsort() + 1
    score_rrf = 1 / (k_rrf + rangs_visu) + 1 / (k_rrf + rangs_meta)

    ordre = np.argsort(-score_rrf)[:k]
    resultats = index.iloc[ordre].copy()
    resultats["score_rrf"] = score_rrf[ordre]
    resultats["sim_visuelle"] = sims_visu[ordre]
    resultats["sim_metadonnees"] = sims_meta[ordre]
    return resultats


for requete in ["gravures de Solis", "editions imprimees a Lyon", "Cadmus combat un serpent", "the tower of Babel"]:
    print(f"\n=== {requete} ===")
    r = rechercher_hybride(requete, k=5)
    for _, row in r.iterrows():
        if pd.notna(row.get("theme_precis")):
            titre = row["theme_precis"]
        elif pd.notna(row.get("theme")):
            titre = row["theme"]
        elif pd.notna(row.get("titre")):
            titre = row["titre"]
        else:
            titre = "?"
        print(f"  rrf={row['score_rrf']:.4f}  visu={row['sim_visuelle']:.3f}  meta={row['sim_metadonnees']:.3f}  "
              f"[{row['source']}] {titre}")

Index : 2563 illustrations, X_index (2563, 768), X_index_meta (2563, 384)


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]


=== gravures de Solis ===
  rrf=0.0238  visu=0.125  meta=0.857  [ovide] P. Ovidii Metamorphosis, Oder Wunderbarliche und seltzame Beschreibung
  rrf=0.0237  visu=0.121  meta=0.857  [ovide] P. Ovidii Metamorphosis, Oder Wunderbarliche und seltzame Beschreibung
  rrf=0.0217  visu=0.117  meta=0.857  [ovide] P. Ovidii Metamorphosis, Oder Wunderbarliche und seltzame Beschreibung
  rrf=0.0215  visu=0.122  meta=0.857  [ovide] P. Ovidii Metamorphosis, Oder Wunderbarliche und seltzame Beschreibung
  rrf=0.0204  visu=0.115  meta=0.857  [ovide] P. Ovidii Metamorphosis, Oder Wunderbarliche und seltzame Beschreibung

=== editions imprimees a Lyon ===
  rrf=0.0187  visu=0.068  meta=0.852  [ovide] Les Oeuvres d’Ovide par Monsieur de Martignac
  rrf=0.0178  visu=0.072  meta=0.820  [ovide] ?
  rrf=0.0176  visu=0.071  meta=0.820  [ovide] ?
  rrf=0.0175  visu=0.071  meta=0.821  [ovide] Les Metamorphoses d'Ovide... par Du Ryer
  rrf=0.0173  visu=0.074  meta=0.813  [ovide] Les Métamorphoses d´Ovide tradui

  rrf=0.0328  visu=0.133  meta=0.843  [bible] Babel
  rrf=0.0318  visu=0.126  meta=0.843  [bible] Babel
  rrf=0.0305  visu=0.120  meta=0.843  [bible] Babel
  rrf=0.0300  visu=0.125  meta=0.843  [bible] Babel
  rrf=0.0299  visu=0.120  meta=0.843  [bible] Babel


## Bilan — index metadonnees

Deux index paralleles : `embedding` (SigLIP, contenu visuel) et
`embedding_metadonnees` (e5, faits bibliographiques/thematiques). Fusion par
**Reciprocal Rank Fusion** plutot qu'une moyenne ponderee des scores bruts — les
deux modeles ont des echelles de similarite tres differentes (SigLIP texte-image
~0.1-0.2, e5 ~0.75-0.85), la RRF ne compare que des rangs, pas des valeurs
absolues, donc insensible a ce probleme d'echelle.

**Limite assumee** : la fusion hybride ne s'applique qu'aux requetes texte —
pour une requete image, on n'a pas de texte a comparer a l'index metadonnees
(il faudrait generer une legende de l'image d'abord, non fait ici). Les requetes
image continuent a utiliser uniquement `embedding` (SigLIP).